# Forensic Multi-Domain YOLO — Analysis & Paper Figures

Run **after** `01_full_training_pipeline.ipynb` completes. Generates all figures and LaTeX tables:
- Unified comparison table (Table I)
- Baseline vs Enhanced change tables (Tables II, III)
- Cross-domain performance table (new Table IV)
- Pareto efficiency plot
- Training convergence curves
- Model complexity bar chart
- LaTeX snippets ready to paste

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'figure.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
})

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =============================================================================
# PATHS — must match the training notebook
# =============================================================================
OUTPUT_ROOT = "/content/drive/MyDrive/ForensicsDataset/rerun_experiments_v2"
FIGURES_DIR = os.path.join(OUTPUT_ROOT, "paper_figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

MODELS_ORDER = [
    "yolov8m_baseline", "yolov8m_enhanced",
    "yolo11m_baseline", "yolo11m_enhanced",
]
DISPLAY_NAMES = {
    "yolov8m_baseline": "YOLOv8-m Base",
    "yolov8m_enhanced": "YOLOv8-m Enh.",
    "yolo11m_baseline": "YOLO11-m Base",
    "yolo11m_enhanced": "YOLO11-m Enh.",
}
COLORS = {
    "yolov8m_baseline": "#1f77b4",
    "yolov8m_enhanced": "#ff7f0e",
    "yolo11m_baseline": "#2ca02c",
    "yolo11m_enhanced": "#d62728",
}
MARKERS = {
    "yolov8m_baseline": "o",
    "yolov8m_enhanced": "s",
    "yolo11m_baseline": "^",
    "yolo11m_enhanced": "D",
}
SEEDS = [42, 123, 7]

## 1. Load Results

In [ ]:
eval_df = pd.read_csv(os.path.join(OUTPUT_ROOT, "all_evaluation_results.csv"))
profiles_df = pd.read_csv(os.path.join(OUTPUT_ROOT, "model_profiles.csv"))

print(f"Loaded {len(eval_df)} evaluation rows and {len(profiles_df)} model profiles.")
display(eval_df.head(12))
display(profiles_df)

## 2. Unified Comparison Table (Table I)

In [ ]:
agg = eval_df.groupby("model").agg(
    P_mean=("precision", "mean"), P_std=("precision", "std"),
    R_mean=("recall", "mean"), R_std=("recall", "std"),
    m50_mean=("mAP50", "mean"), m50_std=("mAP50", "std"),
    m95_mean=("mAP50_95", "mean"), m95_std=("mAP50_95", "std"),
).round(4)

rows = []
for model in MODELS_ORDER:
    if model not in agg.index:
        continue
    r = agg.loc[model]
    p = profiles_df[profiles_df["model"]==model].iloc[0]
    rows.append({
        "Model": DISPLAY_NAMES[model],
        "Params (M)": p["parameters_M"],
        "GFLOPs": p["GFLOPs"],
        "Inf. (ms)": p["inference_ms_mean"],
        "Precision": f"{r['P_mean']:.3f} +/- {r['P_std']:.3f}",
        "Recall": f"{r['R_mean']:.3f} +/- {r['R_std']:.3f}",
        "mAP@50": f"{r['m50_mean']:.3f} +/- {r['m50_std']:.3f}",
        "mAP@50-95": f"{r['m95_mean']:.3f} +/- {r['m95_std']:.3f}",
    })

table_df = pd.DataFrame(rows)
table_df.to_csv(os.path.join(FIGURES_DIR, "table_unified_comparison.csv"), index=False)
display(table_df)

## 3. Cross-Domain Performance Table (new Table IV)

In [ ]:
cd_agg = eval_df.groupby("model").agg(
    overall_mean=("mAP50", "mean"), overall_std=("mAP50", "std"),
    cs_mean=("mAP50_crime_scene", "mean"), cs_std=("mAP50_crime_scene", "std"),
    sd_mean=("mAP50_smart_device", "mean"), sd_std=("mAP50_smart_device", "std"),
).round(4)

cd_rows = []
for model in MODELS_ORDER:
    if model not in cd_agg.index:
        continue
    r = cd_agg.loc[model]
    cd_rows.append({
        "Model": DISPLAY_NAMES[model],
        "Overall mAP@50": f"{r['overall_mean']:.3f} +/- {r['overall_std']:.3f}",
        "Crime Scene": f"{r['cs_mean']:.3f} +/- {r['cs_std']:.3f}",
        "Smart Device": f"{r['sd_mean']:.3f} +/- {r['sd_std']:.3f}",
        "Domain Gap": f"{abs(r['cs_mean'] - r['sd_mean']):.3f}",
    })

cd_table = pd.DataFrame(cd_rows)
cd_table.to_csv(os.path.join(FIGURES_DIR, "table_cross_domain.csv"), index=False)
display(cd_table)

## 4. Pareto Efficiency Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

points = []
for model in MODELS_ORDER:
    p_row = profiles_df[profiles_df["model"] == model]
    if p_row.empty:
        continue
    gflops = p_row["GFLOPs"].values[0]
    model_evals = eval_df[eval_df["model"] == model]
    map95_mean = model_evals["mAP50_95"].mean()
    map95_std = model_evals["mAP50_95"].std()

    ax.errorbar(
        gflops, map95_mean, yerr=map95_std,
        fmt=MARKERS[model], color=COLORS[model],
        markersize=10, capsize=4, linewidth=1.5,
        label=DISPLAY_NAMES[model], zorder=5,
    )
    points.append((gflops, map95_mean))

# Pareto frontier
sorted_pts = sorted(points, key=lambda x: x[0])
pareto = []
max_y = -1
for x, y in sorted_pts:
    if y > max_y:
        pareto.append((x, y))
        max_y = y
if len(pareto) > 1:
    px, py = zip(*pareto)
    ax.plot(px, py, '--', color='gray', alpha=0.5, linewidth=1, zorder=1)

ax.set_xlabel("GFLOPs", fontsize=12)
ax.set_ylabel("mAP@50-95", fontsize=12)
ax.set_title("Pareto Efficiency: Accuracy vs. Computational Cost", fontsize=13)
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.3)

fig.savefig(os.path.join(FIGURES_DIR, "pareto_efficiency.png"), dpi=300)
fig.savefig(os.path.join(FIGURES_DIR, "pareto_efficiency.pdf"))
plt.show()
print(f"Saved to {FIGURES_DIR}/pareto_efficiency.png")

## 5. Training Convergence Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
metrics_to_plot = [
    ("train/box_loss", "Box Loss (Train)", axes[0, 0]),
    ("train/cls_loss", "Cls Loss (Train)", axes[0, 1]),
    ("metrics/mAP50(B)", "mAP@50 (Val)", axes[1, 0]),
    ("metrics/mAP50-95(B)", "mAP@50-95 (Val)", axes[1, 1]),
]

for model in MODELS_ORDER:
    csv_path = os.path.join(OUTPUT_ROOT, model, "seed_42", "train", "results.csv")
    if not os.path.exists(csv_path):
        print(f"[SKIP] {model}: results.csv not found")
        continue

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    for col_name, title, ax in metrics_to_plot:
        if col_name in df.columns:
            ax.plot(
                df["epoch"], df[col_name],
                label=DISPLAY_NAMES[model],
                color=COLORS[model], linewidth=1.2
            )

for _, title, ax in metrics_to_plot:
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Training Convergence (seed=42)", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "training_curves.png"), dpi=300)
fig.savefig(os.path.join(FIGURES_DIR, "training_curves.pdf"))
plt.show()

## 6. Model Complexity Comparison

In [ ]:
models = [DISPLAY_NAMES[m] for m in MODELS_ORDER if m in profiles_df["model"].values]
params = [profiles_df[profiles_df["model"]==m]["parameters_M"].values[0]
          for m in MODELS_ORDER if m in profiles_df["model"].values]
gflops = [profiles_df[profiles_df["model"]==m]["GFLOPs"].values[0]
          for m in MODELS_ORDER if m in profiles_df["model"].values]

x = np.arange(len(models))
width = 0.35

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.bar(x - width/2, params, width, label="Parameters (M)", color="#4c72b0")
ax1.set_ylabel("Parameters (M)", color="#4c72b0")
ax1.tick_params(axis='y', labelcolor="#4c72b0")

ax2 = ax1.twinx()
ax2.bar(x + width/2, gflops, width, label="GFLOPs", color="#dd8452")
ax2.set_ylabel("GFLOPs", color="#dd8452")
ax2.tick_params(axis='y', labelcolor="#dd8452")

ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=15, ha="right")
ax1.set_title("Model Complexity Comparison")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "model_complexity.png"), dpi=300)
fig.savefig(os.path.join(FIGURES_DIR, "model_complexity.pdf"))
plt.show()

## 7. Generate LaTeX Table Snippets

In [ ]:
output_path = os.path.join(FIGURES_DIR, "latex_tables.tex")

agg = eval_df.groupby("model").agg(
    P_m=("precision", "mean"), P_s=("precision", "std"),
    R_m=("recall", "mean"), R_s=("recall", "std"),
    m50_m=("mAP50", "mean"), m50_s=("mAP50", "std"),
    m95_m=("mAP50_95", "mean"), m95_s=("mAP50_95", "std"),
)

with open(output_path, "w") as f:
    # Table I: Unified comparison
    f.write("% TABLE I: Unified Model Comparison\n")
    f.write("\\begin{table}[!htb]\n")
    f.write("\\caption{Unified Performance Comparison (mean $\\pm$ std, 3 seeds)}\n")
    f.write("\\centering\\small\n")
    f.write("\\begin{tabular}{lcccccc}\n\\hline\n")
    f.write("Model & Params & GFLOPs & Precision & Recall & mAP@50 & mAP@50--95 \\\\\n\\hline\n")

    for model in MODELS_ORDER:
        if model not in agg.index:
            continue
        r = agg.loc[model]
        p = profiles_df[profiles_df["model"]==model].iloc[0]
        name = DISPLAY_NAMES[model]
        f.write(
            f"{name} & {p['parameters_M']}M & {p['GFLOPs']} & "
            f"{r['P_m']:.3f}$\\pm${r['P_s']:.3f} & "
            f"{r['R_m']:.3f}$\\pm${r['R_s']:.3f} & "
            f"{r['m50_m']:.3f}$\\pm${r['m50_s']:.3f} & "
            f"{r['m95_m']:.3f}$\\pm${r['m95_s']:.3f} \\\\\n"
        )

    f.write("\\hline\n\\end{tabular}\n")
    f.write("\\label{tab:unified}\n\\end{table}\n\n")

    # Baseline vs Enhanced change tables
    for family, base, enh in [
        ("YOLOv8-m", "yolov8m_baseline", "yolov8m_enhanced"),
        ("YOLO11-m", "yolo11m_baseline", "yolo11m_enhanced"),
    ]:
        if base not in agg.index or enh not in agg.index:
            continue
        b, e = agg.loc[base], agg.loc[enh]
        f.write(f"% TABLE: {family} Baseline vs Enhanced\n")
        f.write("\\begin{table}[!htb]\n")
        f.write(f"\\caption{{{family} Baseline vs Enhanced Performance}}\n")
        f.write("\\centering\n")
        f.write("\\begin{tabular}{lccc}\n\\hline\n")
        f.write("Metric & Baseline & Enhanced & Change \\\\\n\\hline\n")
        for metric, col_m, col_s in [
            ("Precision", "P_m", "P_s"),
            ("Recall", "R_m", "R_s"),
            ("mAP@50", "m50_m", "m50_s"),
            ("mAP@50--95", "m95_m", "m95_s"),
        ]:
            diff = e[col_m] - b[col_m]
            sign = "+" if diff >= 0 else ""
            f.write(
                f"{metric} & {b[col_m]:.3f}$\\pm${b[col_s]:.3f} & "
                f"{e[col_m]:.3f}$\\pm${e[col_s]:.3f} & "
                f"{sign}{diff:.3f} \\\\\n"
            )
        f.write("\\hline\n\\end{tabular}\n\\end{table}\n\n")

print(f"LaTeX tables saved: {output_path}")
# Show the file
with open(output_path) as f:
    print(f.read())

## 8. Summary

In [ ]:
print(f"""
{'='*70}
ALL FIGURES AND TABLES GENERATED
{'='*70}
Saved to: {FIGURES_DIR}/

Figures:
  pareto_efficiency.png/pdf    — Pareto frontier plot
  training_curves.png/pdf      — Loss and mAP over epochs
  model_complexity.png/pdf     — Params and GFLOPs bar chart

Tables:
  table_unified_comparison.csv — Master comparison (Table I)
  table_cross_domain.csv       — Per-domain mAP (Table IV)
  latex_tables.tex             — Ready-to-paste LaTeX for all tables

Copy the .pdf figures into your LaTeX figures/ folder and
\\input the .tex snippets directly into your paper.
{'='*70}
""")